# Detección de Anomalías en Paneles Solares con YOLOv9
Este notebook contiene el pipeline End-to-End para entrenar, predecir y desplegar un modelo de detección de anomalías usando la arquitectura YOLOv9 y el dataset ThermoSolar-PV.

## 1. Instalación de Dependencias
Instalamos las librerías necesarias. Nota: Si estás en Windows y tienes una tarjeta NVIDIA, asegúrate de instalar la versión de PyTorch compatible con CUDA.

In [ ]:
!pip install ultralytics fastapi uvicorn python-multipart opencv-python-headless
# Para PyTorch con CUDA 11.8 (descomentar si es necesario):
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

## 2. Configuración del Dataset
Creamos el archivo `solar_data.yaml` necesario para que Ultralytics encuentre las carpetas de imágenes y etiquetas.

In [ ]:
yaml_content = """# Dataset Path Configurations
path: ImageSet  # Ruta base relativa al directorio actual
train: train/images  # Ruta de imágenes de entrenamiento
val: valid/images    # Ruta de imágenes de validación
test: test/images    # Ruta de imágenes de prueba (opcional)

# Classes
names:
  0: Single Hotspot
  1: Multi Hotspots
  2: Single Diode
  3: Multi Diode
  4: Single Bypassed Substring
  5: Multi Bypassed Substring
  6: String Open Circuit
  7: String Reversed Polarity
"""

with open('solar_data.yaml', 'w') as f:
    f.write(yaml_content)

print("[INFO] solar_data.yaml creado exitosamente.")

## 3. Entrenamiento del Modelo
Usaremos el modelo `yolov9s.pt` por ser ligero y óptimo para gráficas con memoria limitada (ej. RTX 3050 de 4GB VRAM).

In [ ]:
import torch
from ultralytics import YOLO

# Verificar GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[INFO] Entrenando en: {device}")

# Cargar modelo
model = YOLO('yolov9s.pt')

# Iniciar entrenamiento
results = model.train(
    data="solar_data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    device=device,
    workers=2,
    name="yolov9_solar_anomalies_nb"
)

print("[INFO] Entrenamiento finalizado.")